[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/templates/28_moe.ipynb)

# 🔴 Hard: Mixture of Experts (MoE)

*Attention & Transformers*
Implement a **top-k mixture-of-experts** layer as an `nnx.Module`, returning
both the output and the load-balancing auxiliary loss.

### Signature
```python
class MixtureOfExperts(nnx.Module):
    def __init__(self, d_model, d_hidden, num_experts, top_k=2, *, rngs):
        ...
    def __call__(self, x):
        ...  # (B, T, d_model) -> (output, aux_loss)
```

Each expert is an independent 2-layer MLP `d_model -> d_hidden -> d_model` with
a ReLU. The router is a single `(d_model, num_experts)` matrix.

### The routing
1. `logits = x @ W_router` → `(N, E)` for `N = B*T` tokens
2. `probs = softmax(logits)` over **all** experts — used by the aux loss
3. Select the top-$k$ experts per token
4. Renormalise **over the selected $k$ only**, so their weights sum to 1
5. Output is the weighted sum of those $k$ experts' outputs

### The auxiliary loss
$$\mathcal{L}_{\text{aux}} = E \sum_{e=1}^{E} f_e \cdot P_e,
\qquad
f_e = \frac{1}{N}\sum_{n=1}^{N} \mathbb{1}\!\left[e \in \mathrm{top}\text{-}k(n)\right],
\qquad
P_e = \frac{1}{N}\sum_{n=1}^{N} \mathrm{softmax}(\text{logits}_n)_e$$

$f_e$ is the **fraction of tokens** routed to expert $e$ (hard, from top-$k$)
and $P_e$ is the **mean router probability** for expert $e$ (soft, from the full
softmax over all $E$).

Watch the normalisation, because it is a classic follow-up. Each token is
counted once per selected expert, so $\sum_e f_e = k$ while $\sum_e P_e = 1$.
That gives, for this (Switch/Mixtral) convention:

| routing | $\mathcal{L}_{\text{aux}}$ |
|---|---|
| perfectly uniform ($f_e = k/E$, $P_e = 1/E$) | $E \cdot E \cdot \tfrac{k}{E}\cdot\tfrac{1}{E} = k$ |
| total collapse (one expert takes everything) | $E$ |

The famous "uniform gives exactly 1" is the $k = 1$ Switch Transformer case.
DeepSeek-style implementations divide $f_e$ by $k$ so the uniform value is 1 for
any $k$; this task uses the un-divided form, which is what
`transformers`' Mixtral loss computes.

### Rules
- Renormalise over the selected experts only — this is the step people miss
- Return `(output, aux_loss)`; output shape matches the input
- The aux loss must be differentiable **through $P_e$** (the hard counts $f_e$
  are not differentiable, and that is fine — the gradient flows via $P$)
- Store the parameters under these names, since the tests read and overwrite
  them: `self.w_router` `(d_model, num_experts)`, and the experts **stacked on a
  leading expert axis** as `self.w1` `(num_experts, d_model, d_hidden)` and
  `self.w2` `(num_experts, d_hidden, d_model)`. Also keep `self.top_k`.

### Why the aux loss is not optional
Routing is a positive feedback loop: an expert that is slightly better early
gets more tokens, so it trains faster, so it gets picked more. Left alone this
collapses — a handful of experts take nearly all traffic and the rest are dead
weight, so you have paid for $E$ experts and are effectively running two.

The $f_e \cdot P_e$ product is a neat piece of design. $f$ is what you actually
care about but has no gradient (it comes from an argmax). $P$ is differentiable
but does not directly measure load. Multiplying them gives a term whose gradient
pushes down the router probability for experts that are *currently* overloaded,
with the load entering as a constant multiplier.

### Parameters vs FLOPs — the whole point
An MoE layer holds $E$ experts' worth of parameters but activates only $k$ per
token. With $E=64, k=2$ you get 32x the parameters at ~2 experts' compute.
Since capability scales with parameter count while cost scales with *active*
parameters, MoE buys capacity for cheap.

What it costs is memory and communication: all $E$ experts must be resident even
though most are idle for any given token, and at scale the experts are sharded
across devices so routing becomes an all-to-all — which is why real
implementations obsess over expert *capacity* and token dropping, and why the
naive dense-compute version below is correct but not fast.

In [ ]:
# Install jax-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q jax-judge flax')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp
from flax import nnx

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

import jax
import jax.numpy as jnp
from flax import nnx


class MixtureOfExperts(nnx.Module):
    """Top-k mixture of experts. Returns (output, aux_loss)."""

    def __init__(self, d_model: int, d_hidden: int, num_experts: int,
                 top_k: int = 2, *, rngs: nnx.Rngs):
        # Expected attributes: self.top_k, self.w_router (d_model, num_experts),
        # self.w1 (num_experts, d_model, d_hidden), self.w2 (num_experts,
        # d_hidden, d_model).
        pass  # Replace this

    def __call__(self, x):
        """(B, T, d_model) -> ((B, T, d_model), scalar aux_loss)"""
        pass  # Replace this

In [ ]:
# 🔍 Scratch cell — poke at your implementation
import jax
import jax.numpy as jnp
from flax import nnx

layer = MixtureOfExperts(d_model=32, d_hidden=64, num_experts=8, top_k=2, rngs=nnx.Rngs(0))
x = jax.random.normal(jax.random.key(1), (2, 16, 32))

out, aux = layer(x)
print("out:", out.shape, " aux_loss:", float(aux))
print(f"(perfectly uniform routing -> top_k = {layer.top_k}; "
      f"total collapse -> num_experts = {layer.num_experts})")

# How many parameters are there vs how many run per token?
p = nnx.state(layer, nnx.Param)
total = sum(v.size for v in jax.tree.leaves(p))
per_token = 32 * 64 * 2 * 2      # top_k experts, two matrices each
print(f"total expert params: {total}, active per token: ~{per_token}")

In [ ]:
# ✅ SUBMIT — run this cell to check your solution
from jax_judge import check, hint, solution

check("moe")

# hint("moe")      # stuck? nudge without the answer
# solution("moe")  # spoiler: the reference implementation